# Import Libraries & Dependencies

In [1]:
!pip install -q nltk rouge-score pycocoevalcap

In [ ]:
import json
from pathlib import Path
from tqdm.notebook import tqdm
import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk
import pandas as pd
from datasets import Dataset, DatasetDict

from lora import inject_lora
from generate import generate_batch
import evaluate

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'code' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

gptModel = 'gpt2-large'

DATASET_FILES = {
    'train': DATA_DIR / 'trainset.csv',
    'validation': DATA_DIR / 'devset.csv',
    'test': DATA_DIR / 'testset_w_refs.csv',
}
LORA_WEIGHTS_FILE = RESULTS_DIR / gptModel + '_lora_weights.pt'
PREDICTIONS_FILE = RESULTS_DIR / gptModel + '_predictions.txt'
OUTPUT_FILE = RESULTS_DIR / gptModel + '_results.json'

# Table 11
BEAM_SIZE = 10      
LENGTH_PENALTY = 0.9      
NO_REPEAT_NGRAM = 4       
MAX_NEW_TOKENS = 128
GEN_BATCH_SIZE = 8


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lukes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lukes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Load Datasets

In [ ]:
def load_e2e(gptModel):
    base = "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/"
    tokenizer = GPT2Tokenizer.from_pretrained(gptModel)
    tokenizer.pad_token = tokenizer.eos_token

    def rename(df):
        return df.rename(columns={"mr": "input", "ref": "label"})

    def tokenize(batch):
        return tokenizer(
            batch["input"],
            text_target=batch["label"],
            truncation=True,
            max_length=512,
        )

    source_files = {
        'train': 'trainset.csv',
        'validation': 'devset.csv',
        'test': 'testset_w_refs.csv',
    }

    for split, filename in source_files.items():
        csv_path = DATASET_FILES[split]
        if not csv_path.exists():
            pd.read_csv(base + filename).to_csv(csv_path, index=False)

    dataset = DatasetDict({
        "train": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['train']))),
        "validation": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['validation']))),
        "test": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['test']))),
    })

    return dataset.map(tokenize, batched=True)

dataset = load_e2e(gptModel)
print(f"Dataset files stored in: {DATA_DIR}")


Map:   0%|          | 0/42061 [00:00<?, ? examples/s]

Map:   0%|          | 0/4672 [00:00<?, ? examples/s]

Map:   0%|          | 0/4693 [00:00<?, ? examples/s]

## Inject LoRA

In [ ]:
#Load model and Inject LoRA
model = GPT2LMHeadModel.from_pretrained(gptModel)
model = inject_lora(model, rank=4, alpha=32)
model.eval()
print(f"LoRA injected successfully")

# Count how many trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

LoRA injected successfully
trainable params: 393,216


## Training

In [ ]:
# CODE TO TRAIN LORA GPT2 MODEL
tokenizer = GPT2Tokenizer.from_pretrained(gptModel)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def preprocess(batch):
    input_ids, labels = [], []
    for prompt, target in zip(batch["input"], batch["label"]):
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        input_ids.append((prompt_ids + target_ids)[:512])
        labels.append(([-100] * len(prompt_ids) + target_ids)[:512])
    return {"input_ids": input_ids, "labels": labels}

train_dataset = dataset["train"].map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

args = transformers.TrainingArguments(
    output_dir=str(RESULTS_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,        
    weight_decay=0.01,
    warmup_steps=500,         
    label_smoothing_factor=0.1,  
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = transformers.Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=transformers.DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

trainer.train()
torch.save(
    {name: param.detach().cpu() for name, param in model.named_parameters() if param.requires_grad},
    LORA_WEIGHTS_FILE,
)
print(f"Saved LoRA weights to: {LORA_WEIGHTS_FILE}")
model.eval()


Map:   0%|          | 0/42061 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,2.310700
100,1.659800
150,1.562000
200,1.536700
250,1.486400
300,1.488800
350,1.440000


KeyboardInterrupt: 

# Generate Outputs For Testing

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer.padding_side = 'left'

# Load raw test prompts and references
prompts = dataset['test']['input']
references = [[label] for label in dataset['test']['label']]

print(f'Loaded {len(prompts)} test prompts')
print(f'Sample prompt: {prompts[0][:80]}...')


In [ ]:
# Run generation over all test prompts
all_predictions = []
for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc='Generating'):
    raw_batch = prompts[i : i + GEN_BATCH_SIZE]
    # Tokenize the raw prompt strings into the dict that generate_batch expects.
    # generate_batch requires keys "input_ids" and "attention_mask".
    batch = tokenizer(
        raw_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    # generate_batch expects plain Python lists (it calls torch.tensor internally)
    batch = {k: v.tolist() for k, v in batch.items()}
    all_predictions.extend(generate_batch(model, tokenizer, batch, device))

print(f'\nGenerated {len(all_predictions)} predictions')
print('\nSample outputs:')
for i in range(min(3, len(all_predictions))):
    print(f'  [{i}] Prompt: {prompts[i][:60]}...')
    print(f'       Output: {all_predictions[i]}')
    print()

In [ ]:
# Save predictions
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
    for pred in all_predictions:
        f.write(pred + '\n')

print(f'Saved predictions to: {PREDICTIONS_FILE}')


# Evaluating Test Results

Computes all five NLG metrics from Table 3 of the paper:

| Metric  | What it measures |
|---------|-----------------|
| BLEU    | N-gram precision (1–4) with brevity penalty |
| NIST    | Like BLEU but weights rarer n-grams more heavily |
| METEOR  | Unigram F-score with stemming + WordNet synonyms (needs `nltk wordnet`) |
| ROUGE-L | Longest common subsequence F-score |
| CIDEr   | TF-IDF-weighted n-gram cosine similarity |

In [ ]:
print('Computing metrics...')
bleu = evaluate.compute_bleu(all_predictions, references);    print(f'  BLEU:    {bleu}')
nist = evaluate.compute_nist(all_predictions, references);    print(f'  NIST:    {nist}')
meteor = evaluate.compute_meteor(all_predictions, references);  print(f'  METEOR:  {meteor}')
rouge_l = evaluate.compute_rouge_l(all_predictions, references); print(f'  ROUGE-L: {rouge_l}')
cider = evaluate.compute_cider(all_predictions, references);   print(f'  CIDEr:   {cider}')

In [ ]:
results = {
    'num_examples': len(all_predictions),
    'predictions_file': str(PREDICTIONS_FILE),
    'lora_weights_file': str(LORA_WEIGHTS_FILE),
    'BLEU': bleu,
    'NIST': nist,
    'METEOR': meteor,
    'ROUGE-L': rouge_l,
    'CIDEr': cider,
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to: {OUTPUT_FILE}')